# 🦁 Serve a big GGUF model on free Colab GPU → use it in A2I

Notebook នេះ **រត់ model GGUF ធំ** (ឧ. `DavidAU/Qwen3.6-27B-Fable-Fusion-…-GGUF`) លើ GPU ឥតគិតថ្លៃរបស់ Colab
ហើយបើក **OpenAI-compatible API** + **public URL** ដើម្បីឲ្យ **A2I web** ភ្ជាប់ជា "ខួរក្បាល" បាន —
model ល្បីខ្លាំង គ្មាន API បង់ប្រាក់។

**ត្រូវការ**៖ Runtime → Change runtime type → **T4 GPU** (ឬ A100 បើមាន Colab Pro)។

| GPU | Model 27B IQ4 (~17GB) |
| --- | --- |
| T4 (15GB) ឥតគិតថ្លៃ | ដំណើរការបាន តែ **split CPU+GPU** — យឺតបន្តិច |
| A100 (40GB) Pro | លឿន ពេញ GPU |

បើ 27B យឺតពេក ប្តូរ `FILENAME` ទៅ quant តូចជាង (Q3/Q2) នៅ Config ខាងក្រោម។

## ១ — ពិនិត្យ GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "គ្មាន GPU — Runtime → Change runtime type → T4 GPU"
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory/1e9, 1)
print("GPU:", name, "| VRAM:", vram, "GB")

## ២ — Config (កែត្រង់នេះ)

- `REPO_ID` — Hugging Face repo នៃ GGUF
- `FILENAME` — ទុក `None` សិន រួច run cell ជំហាន ៣ ដើម្បីមើលឈ្មោះ file ទាំងអស់ រួចមក paste មកវិញ
- `N_GPU_LAYERS` — ចំនួន layer ដាក់លើ GPU។ T4: ~`24` (split); A100: `-1` (ទាំងអស់)

In [ ]:
REPO_ID = "DavidAU/Qwen3.6-27B-Fable-Fusion-711-Uncensored-Heretic-NM-DAU-NEO-MAX-MTP-GGUF"
FILENAME = None          # ដាក់ឈ្មោះ .gguf ពិតប្រាកដ បន្ទាប់ពី run cell ៣
N_GPU_LAYERS = 24        # T4: ~24 (split CPU+GPU) | A100: -1 | បើ OOM បន្ថយ
N_CTX = 4096             # context length

## ៣ — មើលឈ្មោះ file GGUF ក្នុង repo (ជ្រើសមួយ)

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import list_repo_files
ggufs = sorted(f for f in list_repo_files(REPO_ID) if f.lower().endswith(".gguf"))
print(f"{len(ggufs)} GGUF file:")
for f in ggufs:
    print(" ", f)
print("\n→ Copy ឈ្មោះមួយ (ឧ. quant IQ4_XS ~17GB) ដាក់ក្នុង FILENAME នៅ cell ២ រួច rerun cell ២")

## ៤ — ដំឡើង llama.cpp (CUDA) — ~៥-៨ នាទី

Compile `llama-cpp-python` ជាមួយ CUDA ដើម្បី offload ទៅ GPU។ (បើ compile យូរ/error អាចប្តូរទៅ
prebuilt wheel — មើល comment)។

In [ ]:
# Compile ជាមួយ CUDA (មាំ តែ ~5-8 នាទី)
!CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install -q "llama-cpp-python[server]"

# ជម្រើស (លឿនជាង, prebuilt) បើ compile ខាងលើ fail — ដោះ comment ១ បន្ទាត់ ត្រូវនឹង CUDA របស់ Colab:
# !pip install -q "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## ៥ — ទាញ model ពី Hugging Face (cache ម្តងក្នុង session)

In [ ]:
assert FILENAME, "កំណត់ FILENAME នៅ cell ២ សិន (មើលបញ្ជីពី cell ៣)"
from huggingface_hub import hf_hub_download
# repo private? ដោះ comment ២ បន្ទាត់៖
# from huggingface_hub import login; login()
MODEL_PATH = hf_hub_download(REPO_ID, FILENAME)
print("Model នៅ:", MODEL_PATH)

## ៦ — បើក server (OpenAI-compatible API នៅ :8000)

រត់ background។ បើ VRAM OOM → បន្ថយ `N_GPU_LAYERS` នៅ cell ២ រួច rerun cell នេះ។

In [ ]:
import subprocess, time, os, requests

# បិទ server ចាស់ បើមាន
os.system("pkill -f llama_cpp.server 2>/dev/null")
time.sleep(2)

log = open("server.log", "w")
proc = subprocess.Popen(
    ["python", "-m", "llama_cpp.server",
     "--model", MODEL_PATH,
     "--n_gpu_layers", str(N_GPU_LAYERS),
     "--n_ctx", str(N_CTX),
     "--host", "0.0.0.0", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT,
)

# រង់ចាំ server ឡើង (loading 17GB អាចយូរ ~1-3 នាទី)
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print("✅ Server ready នៅ http://localhost:8000/v1"); break
    except Exception:
        pass
    time.sleep(2)
else:
    print("⚠️ មិនទាន់ ready — មើល log៖"); print(open("server.log").read()[-2000:])

## ៧ — សាកល្បងក្នុង notebook

In [ ]:
import requests
r = requests.post("http://localhost:8000/v1/chat/completions", json={
    "model": "local",
    "messages": [{"role": "user", "content": "សូមណែនាំខ្លួនជាភាសាខ្មែរ ២ ប្រយោគ"}],
    "max_tokens": 200, "temperature": 0.7,
}, timeout=300)
print(r.json()["choices"][0]["message"]["content"])

## ៨ — បើក public URL → ភ្ជាប់ A2I web

Cloudflare quick tunnel (គ្មាន signup) ផ្តល់ URL សាធារណៈ។ Copy URL `https://xxxx.trycloudflare.com`
រួចបូក `/v1`។

In [ ]:
import subprocess, re, time
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
os.system("pkill -f cloudflared 2>/dev/null"); time.sleep(1)

tl = open("tunnel.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                 stdout=tl, stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    try:
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
        if m: url = m.group(0); break
    except Exception:
        pass
print("🌍 Public API base:", (url + "/v1") if url else "មិនទាន់ — rerun cell នេះ")

## ៩ — ភ្ជាប់ក្នុង A2I web

1. បើក A2I web → **⚙️ Settings → AI providers → Add / Custom (OpenAI-compatible)**
2. **API base**: `https://xxxx.trycloudflare.com/v1` (ពី cell ៨)
3. **API key**: អ្វីក៏បាន (ឧ. `a2i`) — server មិនត្រួតពិនិត្យ
4. **Model**: `local`
5. Save → ជ្រើស brain នេះ → ឆាតជាមួយ 27B model លើ Colab GPU 🎉

⚠️ **ចំណាំ**៖
- Colab free ផ្តាច់ session ~90 នាទី idle → tunnel URL ផ្លាស់ប្តូរ រាល់ពេល rerun។
- កុំបិទ tab Colab នេះ ពេលកំពុងប្រើ។
- URL trycloudflare សាធារណៈ — កុំចែក បើមិនចង់ឲ្យអ្នកផ្សេងប្រើ។ ដើម្បីបិទ៖ Runtime → Disconnect។